In [20]:
import pandas as pd
import numpy as np

# Load product master
products = pd.read_csv(r"D:\Artificial Intelligence\SupplyChainManagementProject\supply-chain-ai\data\raw\product_master.csv")

np.random.seed(42)
n = len(products)

# ── Stock health distribution ──────────────────────────────────────────────
# With avg weekly forecast ~3-4 units, we want realistic mix:
#   ~20% low-stock  (1–6 units)   → triggers reorder alerts
#   ~60% healthy    (7–15 units)  → 2–4 weeks of cover
#   ~20% overstocked (16–30 units) → excess capital tied up
segments = np.random.choice(["low", "healthy", "over"], size=n, p=[0.20, 0.60, 0.20])

current_stock = np.where(
    segments == "low",    np.random.randint(1,  7,  n),
    np.where(
    segments == "over",   np.random.randint(16, 31, n),
                          np.random.randint(7,  16, n)   # healthy
))

# ── Reorder & Safety levels grounded in forecast scale (~3-4 units/week) ──
# Reorder level = ~1.5–2.5 weeks of demand  →  5–10 units
# Safety stock  = ~0.5–1.5 weeks of demand  →  2–6 units
reorder_level = np.random.randint(5, 10, n)
safety_stock  = np.random.randint(2,  7, n)

# Safety stock must always be < reorder level
safety_stock = np.minimum(safety_stock, reorder_level - 1)

# ── Build DataFrame ────────────────────────────────────────────────────────
inventory = pd.DataFrame({
    "Product_ID":    products["Product_ID"],
    "Product_Name":  products["Product_Name"],
    "Category":      products["Category"],
    "Subcategory":   products["Subcategory"],
    "Unit":          products["Unit"],
    "Unit_Cost":     products["Unit_Cost"],
    "Standard_Price":products["Standard_Price"],
    "Current_Stock": current_stock,
    "Reorder_Level": reorder_level,
    "Safety_Stock":  safety_stock
})

# ── Procurement dates: Aug–Sep 2024, skewed toward recent ─────────────────
# Low-stock items likely procured earlier (stock depleted since then)
# Healthy/overstocked items procured more recently
start_aug = pd.Timestamp("2024-08-01")
end_sep   = pd.Timestamp("2024-09-20")

def rand_dates(start, end, size):
    return pd.to_datetime(
        np.random.randint(
            int(start.timestamp()),
            int(end.timestamp()),
            size
        ),
        unit="s"
    ).normalize()   # strips time component cleanly (replaces .date)

# Low-stock: procured Aug 1 – Aug 31 (older → stock depleted)
# Healthy:   procured Aug 15 – Sep 15
# Overstocked: procured Sep 1 – Sep 20 (recent → still high)
low_mask  = segments == "low"
over_mask = segments == "over"
hlth_mask = segments == "healthy"

procurement_dates = pd.Series(index=range(n), dtype="datetime64[ns]")
procurement_dates[low_mask]  = rand_dates(pd.Timestamp("2024-08-01"),
                                           pd.Timestamp("2024-08-31"),
                                           low_mask.sum()).values
procurement_dates[hlth_mask] = rand_dates(pd.Timestamp("2024-08-15"),
                                           pd.Timestamp("2024-09-15"),
                                           hlth_mask.sum()).values
procurement_dates[over_mask] = rand_dates(pd.Timestamp("2024-09-01"),
                                           pd.Timestamp("2024-09-20"),
                                           over_mask.sum()).values

inventory["Last_Procurement_Date"] = procurement_dates.dt.date

# ── Derived fields ─────────────────────────────────────────────────────────
inventory["Inventory_Value"] = (
    inventory["Current_Stock"] * inventory["Unit_Cost"]
).round(2)

# Weeks of cover based on avg forecast demand (3.625 units/week from your stats)
AVG_WEEKLY_DEMAND = 3.625
inventory["Weeks_Of_Cover"] = (
    inventory["Current_Stock"] / AVG_WEEKLY_DEMAND
).round(1)

# Boolean flag for procurement agent / reorder logic
inventory["Needs_Reorder"] = (
    inventory["Current_Stock"] <= inventory["Reorder_Level"]
).astype(int)

# ── Save ───────────────────────────────────────────────────────────────────
out_path = r"D:\Artificial Intelligence\SupplyChainManagementProject\supply-chain-ai\data\raw\inventory_master.csv"
inventory.to_csv(out_path, index=False)

# ── Summary ────────────────────────────────────────────────────────────────
print(inventory[["Product_ID", "Current_Stock", "Reorder_Level",
                  "Safety_Stock", "Stock_Status", "Weeks_Of_Cover",
                  "Needs_Reorder", "Last_Procurement_Date"]].head(10))

print(f"\nTotal Records : {len(inventory)}")
print(f"Needs Reorder : {inventory['Needs_Reorder'].sum()} products")
print("\nStock status distribution:")
print(inventory["Stock_Status"].value_counts())
print("\nCurrent_Stock describe:")
print(inventory["Current_Stock"].describe().round(2))

  Product_ID  Current_Stock  Reorder_Level  Safety_Stock Stock_Status  \
0  PROD00001              7              6             3      healthy   
1  PROD00002             19              8             6         over   
2  PROD00003             15              6             3      healthy   
3  PROD00004             10              9             2      healthy   
4  PROD00005              6              6             2          low   
5  PROD00006              4              7             5          low   
6  PROD00007              6              7             4          low   
7  PROD00008             22              7             5         over   
8  PROD00009             13              7             5      healthy   
9  PROD00010              9              8             5      healthy   

   Weeks_Of_Cover  Needs_Reorder Last_Procurement_Date  
0             1.9              0            2024-09-01  
1             5.2              0            2024-09-18  
2             4.1        

In [79]:
inventory.head()

,Product_ID,Product_Name,Category,Subcategory,Unit,Unit_Cost,Standard_Price,Current_Stock,Reorder_Level,Safety_Stock,Stock_Status,Last_Procurement_Date,Inventory_Value,Weeks_Of_Cover,Needs_Reorder
0,PROD00001,Gamma Apex Gadget,Food,A,pcs,131.74,198.31,7,10,5,healthy,2024-08-19,922.18,1.9,1
1,PROD00002,Ultra Omega Device,Beauty,E,pcs,141.88,339.56,19,6,4,over,2024-09-10,2695.72,5.2,0
2,PROD00003,Alpha Fusion Gadget,Sport,D,pcs,160.18,187.85,15,8,5,healthy,2024-08-31,2402.70,4.1,0
3,PROD00004,Apex Apex Item,Food,D,pcs,178.55,224.00,10,10,6,healthy,2024-09-04,1785.50,2.8,1
4,PROD00005,Fusion Prime Widget,Home,D,pcs,70.91,246.55,6,6,2,low,2024-08-24,425.46,1.7,1


In [9]:
from utils.db_crud import save_to_db

In [10]:
save_to_db(inventory,'inventory_data','master_data')

In [21]:
from utils.db_crud import load_from_db

In [22]:
query = 'select * from master_data.inventory_data;'
inventory = load_from_db(query,'inventory_data')

In [23]:
inventory.head()

,Product_ID,Product_Name,Category,Subcategory,Unit,Unit_Cost,Standard_Price,Current_Stock,Reorder_Level,Safety_Stock,Last_Procurement_Date,Inventory_Value
0,PROD00001,Gamma Apex Gadget,Food,A,pcs,131.74,198.31,7,10,5,2024-08-19,922.18
1,PROD00002,Ultra Omega Device,Beauty,E,pcs,141.88,339.56,19,6,4,2024-09-10,2695.72
2,PROD00003,Alpha Fusion Gadget,Sport,D,pcs,160.18,187.85,15,8,5,2024-08-31,2402.70
3,PROD00004,Apex Apex Item,Food,D,pcs,178.55,224.00,10,10,6,2024-09-04,1785.50
4,PROD00005,Fusion Prime Widget,Home,D,pcs,70.91,246.55,6,6,2,2024-08-24,425.46


In [24]:
query = 'select * from forecast_data.forecast_sales_orders;'
forecast = load_from_db(query,'forecast_data')

In [25]:
forecast.head()

,unique_id,ds,CrostonClassic
0,PROD00001,2024-09-15,4.228454
1,PROD00002,2024-09-01,2.640355
2,PROD00003,2024-09-22,3.648528
3,PROD00004,2024-09-22,3.867649
4,PROD00005,2024-09-29,3.287539


In [26]:
stockout = inventory.merge(
        on=forecast['unique_id'],
        how='left',right=forecast
    )[['unique_id','Current_Stock','Reorder_Level','Safety_Stock','CrostonClassic']]

In [27]:
stockout

,unique_id,Current_Stock,Reorder_Level,Safety_Stock,CrostonClassic
0,PROD00001,7,10,5,4.228454
1,PROD00002,19,6,4,2.640355
2,PROD00003,15,8,5,3.648528
3,PROD00004,10,10,6,3.867649
4,PROD00005,6,6,2,3.287539
...,...,...,...,...,...
195,PROD00196,7,7,5,3.826658
196,PROD00197,9,6,4,3.010561
197,PROD00198,16,8,5,2.629620
198,PROD00199,21,9,3,3.086941


In [35]:
stockout['stockout'] = (stockout['Current_Stock'] - (stockout['CrostonClassic'])).round()

In [34]:
stockout.head()

,unique_id,Current_Stock,Reorder_Level,Safety_Stock,CrostonClassic,stockout
0,PROD00001,7,10,5,4.228454,3.0
1,PROD00002,19,6,4,2.640355,16.0
2,PROD00003,15,8,5,3.648528,11.0
3,PROD00004,10,10,6,3.867649,6.0
4,PROD00005,6,6,2,3.287539,3.0


In [16]:
stockout['stockout'].describe()

count    200.00000
mean       7.61500
std        6.69559
min       -4.00000
25%        3.00000
50%        6.00000
75%       12.00000
max       25.00000
Name: stockout, dtype: float64

In [40]:
import numpy as np

In [51]:
conditions = [

    # Already in safety stock
    stockout['Current_Stock'] <= stockout['Safety_Stock'],

    # Below reorder level
    (
        (stockout['Current_Stock'] > stockout['Safety_Stock']) &
        (stockout['Current_Stock'] <= stockout['Reorder_Level'])
    ),

    # Next week's demand will push inventory into safety stock
    (
        (stockout['Current_Stock'] > stockout['Reorder_Level']) &
        (
            stockout['Current_Stock'] - stockout['CrostonClassic']
            <= stockout['Safety_Stock']
        )
    ),

    # Healthy inventory
    (
        (stockout['Current_Stock'] > stockout['Reorder_Level']) &
        (
            stockout['Current_Stock'] - stockout['CrostonClassic']
            > stockout['Safety_Stock']
        )
    )
]

choices = [
    'CRITICAL',
    'REORDER_NOW', 
    'AT_RISK', 
    'SUFFICIENT'
]

stockout['stockout_label'] = np.select(
    conditions,
    choices,
    default='Healthy'
)

In [67]:
stockout.head(20)

,unique_id,Current_Stock,Reorder_Level,Safety_Stock,CrostonClassic,stockout,stockout_label
0,PROD00001,7,10,5,4.228454,3.0,REORDER_NOW
1,PROD00002,19,6,4,2.640355,16.0,SUFFICIENT
2,PROD00003,15,8,5,3.648528,11.0,SUFFICIENT
3,PROD00004,10,10,6,3.867649,6.0,REORDER_NOW
4,PROD00005,6,6,2,3.287539,3.0,REORDER_NOW
5,PROD00006,4,9,2,3.065903,1.0,REORDER_NOW
6,PROD00007,6,6,5,3.786447,2.0,REORDER_NOW
7,PROD00008,22,10,6,1.969974,20.0,SUFFICIENT
8,PROD00009,13,7,4,3.290359,10.0,SUFFICIENT
9,PROD00010,9,7,2,3.417734,6.0,SUFFICIENT


In [53]:
stockout['stockout_label'].value_counts()

stockout_label
SUFFICIENT     124
REORDER_NOW     41
CRITICAL        26
AT_RISK          9
Name: count, dtype: int64

In [77]:
def label_data(forecast,inventory):
    stockout = inventory.merge(
        on=forecast['unique_id'],
        how='left',right=forecast
    )[['unique_id','Current_Stock','Reorder_Level','Safety_Stock','CrostonClassic']]

    stockout['stockout'] = (stockout['Current_Stock'] - (stockout['CrostonClassic'])).round()

    conditions = [

        # Already in safety stock
        stockout['Current_Stock'] <= stockout['Safety_Stock'],
    
        # Below reorder level
        (
            (stockout['Current_Stock'] > stockout['Safety_Stock']) &
            (stockout['Current_Stock'] <= stockout['Reorder_Level'])
        ),
    
        # Next week's demand will push inventory into safety stock
        (
            (stockout['Current_Stock'] > stockout['Reorder_Level']) &
            (
                stockout['Current_Stock'] - stockout['CrostonClassic']
                <= stockout['Safety_Stock']
            )
        ),
    
        # Healthy inventory
        (
            (stockout['Current_Stock'] > stockout['Reorder_Level']) &
            (
                stockout['Current_Stock'] - stockout['CrostonClassic']
                > stockout['Safety_Stock']
            )
        )
    ]
    
    choices = [
        'CRITICAL',
        'REORDER_NOW', 
        'AT_RISK', 
        'SUFFICIENT'
    ]
    
    stockout['stockout_label'] = np.select(
        conditions,
        choices,
        default='Healthy'
    )

    return stockout 
    

In [78]:
df = label_data(forecast,inventory)

In [79]:
def stock_summary(df):
    summary = {}
    for label in ["CRITICAL", "REORDER_NOW", "AT_RISK", "SUFFICIENT"]:
        subset = df[df["stockout_label"] == label]
        summary[label] = {
            "count":    len(subset),
            "sample":   subset["unique_id"].head(3).tolist(),  # just 3 examples
            "pct":      round(len(subset) / len(df) * 100, 1)
        }
    return summary

summary = stock_summary(df)

In [76]:
summary

{'CRITICAL': {'count': 26,
  'sample': ['PROD00011', 'PROD00016', 'PROD00022'],
  'pct': 13.0},
 'REORDER_NOW': {'count': 41,
  'sample': ['PROD00001', 'PROD00004', 'PROD00005'],
  'pct': 20.5},
 'AT_RISK': {'count': 9,
  'sample': ['PROD00019', 'PROD00021', 'PROD00024'],
  'pct': 4.5},
 'SUFFICIENT': {'count': 124,
  'sample': ['PROD00002', 'PROD00003', 'PROD00008'],
  'pct': 62.0}}

In [59]:
from backend.services.db_service import Load_Data

load_df = Load_Data()

In [60]:
prods = load_df.products

In [61]:
prods.head()

,Product_ID,SKU,Product_Name,Category,Subcategory,Unit,Unit_Cost,Standard_Price,Launch_Date,Discontinuation_Date
0,PROD00001,XAJI0Y6DPB,Gamma Apex Gadget,Food,A,pcs,131.74,198.31,2022-11-18,None
1,PROD00002,HSAHXTHV3A,Ultra Omega Device,Beauty,E,pcs,141.88,339.56,2023-04-13,None
2,PROD00003,3ZMF8MDD4V,Alpha Fusion Gadget,Sport,D,pcs,160.18,187.85,2022-03-16,None
3,PROD00004,30T9NT3W5U,Apex Apex Item,Food,D,pcs,178.55,224.00,2022-01-04,None
4,PROD00005,ZBIKCIDKWN,Fusion Prime Widget,Home,D,pcs,70.91,246.55,2023-10-08,None


In [4]:
prods_list = prods['Product_ID']

In [5]:
import random

In [7]:
obj = SalesSimulator()

In [14]:
sales = load_df.

AttributeError: 'Load_Data' object has no attribute 'processed_sales'

In [10]:
sales.head()

,Order_ID,Customer_ID,Product_ID,Order_Date,Order_Status,Order_Quantity,Unit_Price,Discount,Shipping_Mode,Shipping_Carrier,Shipping_Date_Scheduled,Shipping_Date_Actual,Delivery_Status,Late_Delivery_Risk_Flag,VAT_Rate,COGS,Unit_Price_Effective,Order_Total,VAT_Amount,Profit_Per_Order
0,ORD0000001,CUST00481,PROD00135,2023-06-16,Completed,8,200.54,0.12,Standard,CarrierA,2023-06-21,2023-06-21,On Time,0,0.19,1510.96,176.4752,1411.8016,268.242304,-99.1584
1,ORD0000002,CUST00111,PROD00147,2023-09-23,Pending,3,220.96,0.18,Same Day,CarrierB,2023-09-26,2023-09-26,On Time,0,0.20,371.61,181.1872,543.5616,108.712320,171.9516
2,ORD0000003,CUST00104,PROD00172,2022-12-17,Cancelled,2,15.87,0.09,Economy,CarrierD,2022-12-18,2022-12-20,Late,1,0.20,16.46,14.4417,28.8834,5.776680,12.4234
3,ORD0000004,CUST00316,PROD00063,2023-10-11,Pending,7,191.91,0.06,Express,CarrierC,2023-10-15,2023-10-19,Late,1,0.00,1166.20,180.3954,1262.7678,0.000000,96.5678
4,ORD0000005,CUST00171,PROD00065,2022-10-04,Completed,6,192.00,0.19,Same Day,CarrierB,2022-10-05,2022-10-05,On Time,0,0.20,358.68,155.5200,933.1200,186.624000,574.4400


In [1]:
from backend.services.db_service import Load_Data

load_df = Load_Data()

In [2]:

sales = load_df.processed_sales

In [3]:
sales

,ds,y,unique_id,zero_demand_ratio
0,2022-01-09,3,PROD00001,0.482270
1,2022-01-16,0,PROD00001,0.482270
2,2022-01-23,0,PROD00001,0.482270
3,2022-01-30,7,PROD00001,0.482270
4,2022-02-06,10,PROD00001,0.482270
...,...,...,...,...
28161,2024-09-01,11,PROD00200,0.555556
28162,2024-09-08,0,PROD00200,0.555556
28163,2024-09-15,0,PROD00200,0.555556
28164,2024-09-22,0,PROD00200,0.555556


In [4]:
grouped = sales[['unique_id','zero_demand_ratio']].groupby('unique_id').mean().reset_index()

In [5]:
grouped

,unique_id,zero_demand_ratio
0,PROD00001,0.482270
1,PROD00002,0.492754
2,PROD00003,0.447552
3,PROD00004,0.507042
4,PROD00005,0.506944
...,...,...
195,PROD00196,0.496503
196,PROD00197,0.496403
197,PROD00198,0.573427
198,PROD00199,0.503597


In [6]:
grouped = grouped.rename(columns={"unique_id":"Product_ID"})

In [7]:
grouped.head()

,Product_ID,zero_demand_ratio
0,PROD00001,0.482270
1,PROD00002,0.492754
2,PROD00003,0.447552
3,PROD00004,0.507042
4,PROD00005,0.506944


In [8]:
inventory = load_df.inventory

In [9]:
inventory.head()

,Product_ID,Product_Name,Category,Subcategory,Unit,Unit_Cost,Standard_Price,Current_Stock,Reorder_Level,Safety_Stock,Last_Procurement_Date,Inventory_Value
0,PROD00001,Gamma Apex Gadget,Food,A,pcs,131.74,198.31,7,6,3,01-09-2024,922.18
1,PROD00002,Ultra Omega Device,Beauty,E,pcs,141.88,339.56,19,8,6,18-09-2024,2695.72
2,PROD00003,Alpha Fusion Gadget,Sport,D,pcs,160.18,187.85,15,6,3,02-09-2024,2402.70
3,PROD00004,Apex Apex Item,Food,D,pcs,178.55,224.00,10,9,2,03-09-2024,1785.50
4,PROD00005,Fusion Prime Widget,Home,D,pcs,70.91,246.55,6,6,2,29-08-2024,425.46


In [10]:
new_df = inventory.merge(right=grouped,on='Product_ID',how='left')[['Product_ID','zero_demand_ratio','Current_Stock']]

In [11]:
new_df

,Product_ID,zero_demand_ratio,Current_Stock
0,PROD00001,0.482270,7
1,PROD00002,0.492754,19
2,PROD00003,0.447552,15
3,PROD00004,0.507042,10
4,PROD00005,0.506944,6
...,...,...,...
195,PROD00196,0.496503,7
196,PROD00197,0.496403,9
197,PROD00198,0.573427,16
198,PROD00199,0.503597,21


In [12]:
forecast = load_df.forecast

In [13]:
forecast.head()

,unique_id,ds,CrostonClassic
0,PROD00001,2024-09-15,4.228453
1,PROD00002,2024-09-01,2.640355
2,PROD00003,2024-09-22,3.648528
3,PROD00004,2024-09-22,3.867649
4,PROD00005,2024-09-29,3.287538


In [14]:
forecast = forecast.rename(columns={"unique_id":"Product_ID","CrostonClassic":"forecast"})

In [15]:
products = new_df.merge(on='Product_ID',right=forecast,how='left')[['Product_ID','zero_demand_ratio','Current_Stock','forecast']]

In [16]:
products

,Product_ID,zero_demand_ratio,Current_Stock,forecast
0,PROD00001,0.482270,7,4.228453
1,PROD00002,0.492754,19,2.640355
2,PROD00003,0.447552,15,3.648528
3,PROD00004,0.507042,10,3.867649
4,PROD00005,0.506944,6,3.287538
...,...,...,...,...
195,PROD00196,0.496503,7,3.826658
196,PROD00197,0.496403,9,3.010561
197,PROD00198,0.573427,16,2.629620
198,PROD00199,0.503597,21,3.086941


In [17]:
import numpy as np
import pandas as pd

from utils.logger import logging
from utils.exception import CustomException
import sys


class SalesSimulator:

    def __init__(self, random_state: int = 42):

        self.rng = np.random.default_rng(random_state)

    def generate_sales(self, products: pd.DataFrame) -> pd.DataFrame:
        """
        Generate one day's sales for each SKU.

        Required Columns
        ----------------
        unique_id
        forecast
        current_stock

        Returns
        -------
        unique_id
        sales_qty
        """

        try:

            logging.info("Generating simulated daily sales")

            sales = []

            for _, row in products.iterrows():

                sku = row["Product_ID"]

                stock = int(row["Current_Stock"])

                forecast = float(row["forecast"])

                # No inventory available
                if stock <= 0:

                    qty = 0

                else:

                    # Inventory constraint
                    demand = self.rng.poisson(max(forecast, 0))

                    sales_qty = min(demand, stock)

                    lost_sales = demand - sales_qty

                sales.append(
                    {
                        "Product_ID": sku,
                        "demand": int(demand),
                        "sales_qty":int(sales_qty),
                        "lost_sales":int(lost_sales)
                    }
                )

            logging.info("Daily sales generated successfully")

            return pd.DataFrame(sales)

        except Exception as e:

            logging.critical(
                f"Error while generating sales: {str(e)}"
            )

            raise CustomException(e, sys)

In [18]:
obj = SalesSimulator()

In [19]:
res = obj.generate_sales(products)

In [20]:
res

,Product_ID,demand,sales_qty,lost_sales
0,PROD00001,6,6,0
1,PROD00002,2,2,0
2,PROD00003,5,5,0
3,PROD00004,3,3,0
4,PROD00005,5,5,0
...,...,...,...,...
195,PROD00196,3,3,0
196,PROD00197,4,4,0
197,PROD00198,2,2,0
198,PROD00199,6,6,0


In [21]:
res.describe()

,demand,sales_qty,lost_sales
count,200.000000,200.00000,200.000000
mean,3.605000,3.39000,0.215000
std,2.138185,2.04912,0.996867
min,0.000000,0.00000,0.000000
25%,2.000000,2.00000,0.000000
50%,3.000000,3.00000,0.000000
75%,5.000000,5.00000,0.000000
max,10.000000,10.00000,8.000000


In [22]:
s = load_df.sales_orders

In [23]:
s

,Order_ID,Customer_ID,Product_ID,Order_Date,Order_Status,Order_Quantity,Unit_Price,Discount,Shipping_Mode,Shipping_Carrier,Shipping_Date_Scheduled,Shipping_Date_Actual,Delivery_Status,Late_Delivery_Risk_Flag,VAT_Rate,COGS,Unit_Price_Effective,Order_Total,VAT_Amount,Profit_Per_Order
0,ORD0000001,CUST00481,PROD00135,2023-06-16,Completed,8,200.54,0.12,Standard,CarrierA,2023-06-21,2023-06-21,On Time,0,0.19,1510.96,176.4752,1411.8016,268.242304,-99.1584
1,ORD0000002,CUST00111,PROD00147,2023-09-23,Pending,3,220.96,0.18,Same Day,CarrierB,2023-09-26,2023-09-26,On Time,0,0.20,371.61,181.1872,543.5616,108.712320,171.9516
2,ORD0000003,CUST00104,PROD00172,2022-12-17,Cancelled,2,15.87,0.09,Economy,CarrierD,2022-12-18,2022-12-20,Late,1,0.20,16.46,14.4417,28.8834,5.776680,12.4234
3,ORD0000004,CUST00316,PROD00063,2023-10-11,Pending,7,191.91,0.06,Express,CarrierC,2023-10-15,2023-10-19,Late,1,0.00,1166.20,180.3954,1262.7678,0.000000,96.5678
4,ORD0000005,CUST00171,PROD00065,2022-10-04,Completed,6,192.00,0.19,Same Day,CarrierB,2022-10-05,2022-10-05,On Time,0,0.20,358.68,155.5200,933.1200,186.624000,574.4400
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19995,ORD0019996,CUST00193,PROD00076,2022-08-13,Completed,5,224.88,0.05,Economy,CarrierA,2022-08-19,2022-08-21,Late,1,0.20,735.80,213.6360,1068.1800,213.636000,332.3800
19996,ORD0019997,CUST00079,PROD00109,2024-07-28,Completed,3,169.92,0.13,Same Day,CarrierA,2024-08-03,2024-08-07,Late,1,0.10,148.86,147.8304,443.4912,44.349120,294.6312
19997,ORD0019998,CUST00057,PROD00119,2022-02-16,Pending,4,206.73,0.11,Express,CarrierD,2022-02-21,2022-02-24,Late,1,0.25,716.20,183.9897,735.9588,183.989700,19.7588
19998,ORD0019999,CUST00097,PROD00113,2022-07-04,Completed,2,194.97,0.15,Standard,CarrierA,2022-07-08,2022-07-11,Late,1,0.00,372.58,165.7245,331.4490,0.000000,-41.1310


In [24]:
s['Order_Quantity'].describe()

count    20000.000000
mean         5.010600
std          1.998421
min          1.000000
25%          4.000000
50%          5.000000
75%          6.000000
max         17.000000
Name: Order_Quantity, dtype: float64

In [25]:
print((s["Order_Quantity"] == 0).sum())
print((s["Order_Quantity"] == 0).mean())

0
0.0


In [26]:
products["forecast"].describe()

count    200.000000
mean       3.625271
std        0.796691
min        1.969974
25%        3.063442
50%        3.529592
75%        4.055592
max        5.992811
Name: forecast, dtype: float64

In [28]:
res['sales_qty'].value_counts()

sales_qty
3     39
2     38
1     33
4     31
5     18
6     17
7     11
0      6
8      5
9      1
10     1
Name: count, dtype: int64

In [33]:
new_df = load_df.sales_orders

In [34]:
new_df.head()

,Order_ID,Customer_ID,Product_ID,Order_Date,Order_Status,Order_Quantity,Unit_Price,Discount,Shipping_Mode,Shipping_Carrier,Shipping_Date_Scheduled,Shipping_Date_Actual,Delivery_Status,Late_Delivery_Risk_Flag,VAT_Rate,COGS,Unit_Price_Effective,Order_Total,VAT_Amount,Profit_Per_Order
0,ORD0000001,CUST00481,PROD00135,2023-06-16,Completed,8,200.54,0.12,Standard,CarrierA,2023-06-21,2023-06-21,On Time,0,0.19,1510.96,176.4752,1411.8016,268.242304,-99.1584
1,ORD0000002,CUST00111,PROD00147,2023-09-23,Pending,3,220.96,0.18,Same Day,CarrierB,2023-09-26,2023-09-26,On Time,0,0.20,371.61,181.1872,543.5616,108.712320,171.9516
2,ORD0000003,CUST00104,PROD00172,2022-12-17,Cancelled,2,15.87,0.09,Economy,CarrierD,2022-12-18,2022-12-20,Late,1,0.20,16.46,14.4417,28.8834,5.776680,12.4234
3,ORD0000004,CUST00316,PROD00063,2023-10-11,Pending,7,191.91,0.06,Express,CarrierC,2023-10-15,2023-10-19,Late,1,0.00,1166.20,180.3954,1262.7678,0.000000,96.5678
4,ORD0000005,CUST00171,PROD00065,2022-10-04,Completed,6,192.00,0.19,Same Day,CarrierB,2022-10-05,2022-10-05,On Time,0,0.20,358.68,155.5200,933.1200,186.624000,574.4400


In [35]:
new_df.columns

Index(['Order_ID', 'Customer_ID', 'Product_ID', 'Order_Date', 'Order_Status',
       'Order_Quantity', 'Unit_Price', 'Discount', 'Shipping_Mode',
       'Shipping_Carrier', 'Shipping_Date_Scheduled', 'Shipping_Date_Actual',
       'Delivery_Status', 'Late_Delivery_Risk_Flag', 'VAT_Rate', 'COGS',
       'Unit_Price_Effective', 'Order_Total', 'VAT_Amount',
       'Profit_Per_Order'],
      dtype='object')

## Sales Reset

In [10]:
from backend.services.db_service import Load_Data

load_df = Load_Data()

In [20]:
processed_sales = load_df.load("processed_sales")

In [21]:
processed_sales.head()

,unique_id,ds,y,zero_demand_ratio
0,PROD00001,2022-01-09,3,0.48951
1,PROD00001,2022-01-16,0,0.48951
2,PROD00001,2022-01-23,0,0.48951
3,PROD00001,2022-01-30,7,0.48951
4,PROD00001,2022-02-06,10,0.48951


In [22]:
processed_sales.sort_values('ds',ascending=False)

,unique_id,ds,y,zero_demand_ratio
28267,PROD00200,2024-09-22,0,0.555556
19647,PROD00139,2024-09-22,4,0.429577
567,PROD00004,2024-09-22,4,0.510490
18800,PROD00133,2024-09-22,7,0.475524
18941,PROD00134,2024-09-22,6,0.485915
...,...,...,...,...
28125,PROD00200,2022-01-02,3,0.555556
24591,PROD00175,2022-01-02,4,0.555556
13012,PROD00093,2022-01-02,7,0.500000
12727,PROD00091,2022-01-02,3,0.513889


In [6]:
from utils.db_crud import save_to_db,load_from_db

In [24]:
save_to_db(processed_sales,'simulated_sales','simulation_data')

In [19]:
from backend.pipelines.forecasting_pipeline import reports

r = reports()

In [25]:
labeled = load_df.load("labeled_inventory")

In [26]:
labeled

,unique_id,Current_Stock,Reorder_Level,Safety_Stock,CrostonClassic,stockout,stockout_label
0,PROD00001,7,6,3,3.465758,4.0,SUFFICIENT
1,PROD00002,12,8,6,3.914003,8.0,SUFFICIENT
2,PROD00003,1,6,3,2.416498,-1.0,CRITICAL
3,PROD00004,10,9,2,4.441459,6.0,SUFFICIENT
4,PROD00005,6,6,2,3.767711,2.0,REORDER_NOW
...,...,...,...,...,...,...,...
195,PROD00196,6,7,3,4.299417,2.0,REORDER_NOW
196,PROD00197,6,6,5,2.456148,4.0,REORDER_NOW
197,PROD00198,13,6,5,2.057550,11.0,SUFFICIENT
198,PROD00199,18,7,2,3.324199,15.0,SUFFICIENT


## inventory Reset

In [16]:
import pandas as pd
from backend.services.db_service import Load_Data
from utils.db_crud import save_to_db

load_df = Load_Data()

inventory = load_df.load("inventory")
labeled_inventory = pd.read_csv(r"D:\Artificial Intelligence\supply-chain-ai (copy)\data\inventory_snapshot.csv")

inventory['Current_Stock'] = labeled_inventory['Current_Stock']
inventory['Inventory_Value'] = round(labeled_inventory['Current_Stock'] * inventory['Unit_Cost'],2)
inventory['Last_Planning_Date'] = pd.Timestamp('22-09-2024')
inventory['Last_Updated'] = pd.Timestamp('22-09-2024')

In [17]:
save_to_db(inventory,'inventory_data','master_data')

In [18]:
inventory.head()

,Product_ID,Product_Name,Category,Subcategory,Unit,Unit_Cost,Standard_Price,Current_Stock,Reorder_Level,Safety_Stock,Last_Planning_Date,Inventory_Value,Last_Updated
0,PROD00001,Gamma Apex Gadget,Food,A,pcs,131.74,198.31,7,6,3,2024-09-22,922.18,2024-09-22
1,PROD00002,Ultra Omega Device,Beauty,E,pcs,141.88,339.56,12,8,6,2024-09-22,1702.56,2024-09-22
2,PROD00003,Alpha Fusion Gadget,Sport,D,pcs,160.18,187.85,1,6,3,2024-09-22,160.18,2024-09-22
3,PROD00004,Apex Apex Item,Food,D,pcs,178.55,224.00,10,9,2,2024-09-22,1785.50,2024-09-22
4,PROD00005,Fusion Prime Widget,Home,D,pcs,70.91,246.55,6,6,2,2024-09-22,425.46,2024-09-22


In [10]:
procurements = load_df.load("procurement")

In [11]:
procurements

,PO_ID,Supplier_ID,Product_ID,Qty_To_Order,Unit_Cost,Total_Cost,Order_Date,Order_Status,Expected_Delivery_Date,Actual_Delivery_Date
0,PO00001,SUP00017,PROD00001,9,40.0,360.0,2024-08-09,Delivered,2024-08-13,2024-08-13
1,PO00002,SUP00055,PROD00004,9,77.0,692.0,2024-07-20,Delivered,2024-07-25,2024-07-25
2,PO00003,SUP00021,PROD00006,16,78.0,1250.0,2024-07-01,Delivered,2024-07-06,2024-07-06
3,PO00004,SUP00025,PROD00006,9,37.0,329.0,2024-07-01,Delivered,2024-07-05,2024-07-05
4,PO00005,SUP00062,PROD00006,12,37.0,441.0,2024-07-20,Delivered,2024-07-24,2024-07-24
...,...,...,...,...,...,...,...,...,...,...
171,PO00172,SUP00089,PROD00192,13,79.0,1027.0,2024-07-20,Delivered,2024-07-24,2024-07-24
172,PO00173,SUP00091,PROD00192,7,10.0,70.0,2024-07-20,Delivered,2024-07-25,2024-07-25
173,PO00174,SUP00033,PROD00192,14,74.0,1042.0,2024-08-09,Delivered,2024-08-14,2024-08-14
174,PO00175,SUP00020,PROD00196,9,19.0,167.0,2024-08-09,Delivered,2024-08-14,2024-08-14


In [10]:
import pandas as pd

pro = pd.read_csv(r"D:\Artificial Intelligence\supply-chain-ai (copy)\data\procurements_snapshot.csv")

save_to_db(pro,'procurement_orders','procurements_data')

In [27]:
sim.head()

,unique_id,ds,y
0,PROD00001,2022-01-09,3
1,PROD00001,2022-01-16,0
2,PROD00001,2022-01-23,0
3,PROD00001,2022-01-30,7
4,PROD00001,2022-02-06,10


In [26]:
sim.drop(columns=["Unnamed: 0"], errors="ignore", inplace=True)

In [24]:
sim = load_df.load("simulated_sales")

In [7]:
processed_db = load_df.load("processed_sales")
processed_db

,unique_id,ds,y,zero_demand_ratio
0,PROD00001,2022-01-09,3,0.485915
1,PROD00001,2022-01-16,0,0.485915
2,PROD00001,2022-01-23,0,0.485915
3,PROD00001,2022-01-30,7,0.485915
4,PROD00001,2022-02-06,10,0.485915
...,...,...,...,...
28263,PROD00200,2024-08-25,6,0.559441
28264,PROD00200,2024-09-01,11,0.559441
28265,PROD00200,2024-09-08,0,0.559441
28266,PROD00200,2024-09-15,0,0.559441


In [1]:
import pandas as pd

processed = pd.read_csv(r"D:\Artificial Intelligence\supply-chain-ai (copy)\data\sales_snapshot.csv")

processed.head()

,unique_id,ds,y,zero_demand_ratio
0,PROD00001,2022-01-09,3,0.485915
1,PROD00001,2022-01-16,0,0.485915
2,PROD00001,2022-01-23,0,0.485915
3,PROD00001,2022-01-30,7,0.485915
4,PROD00001,2022-02-06,10,0.485915


In [4]:
processed['ds'].info()

<class 'pandas.core.series.Series'>
RangeIndex: 28268 entries, 0 to 28267
Series name: ds
Non-Null Count  Dtype         
--------------  -----         
28268 non-null  datetime64[ns]
dtypes: datetime64[ns](1)
memory usage: 221.0 KB


In [3]:
processed['ds'] = (pd.to_datetime(processed['ds']))

In [7]:
save_to_db(processed,"processed_sales_orders","processed_data")

In [26]:
processed.sort_values(by='ds',ascending=False)

,unique_id,ds,y,zero_demand_ratio
28267,PROD00200,2024-09-22,0,0.559441
19647,PROD00139,2024-09-22,4,0.425532
567,PROD00004,2024-09-22,4,0.507042
18800,PROD00133,2024-09-22,7,0.471831
18941,PROD00134,2024-09-22,6,0.482270
...,...,...,...,...
28125,PROD00200,2022-01-02,3,0.559441
24591,PROD00175,2022-01-02,4,0.559441
13012,PROD00093,2022-01-02,7,0.496503
12727,PROD00091,2022-01-02,3,0.510490


In [8]:
save_to_db(processed,"simulated_sales","simulation_data")

In [11]:
from backend.services.db_service import Load_Data
from utils.db_crud import save_to_db
import pandas as pd
import numpy as np

# Reproducible randomness
rng = np.random.default_rng(42)

load_df = Load_Data()

procurement = load_df.load("procurement")

# Convert Order_Date to datetime
procurement["Order_Date"] = pd.to_datetime(procurement["Order_Date"])

# Random lead time (4 or 5 days)
lead_time = rng.integers(4, 6, size=len(procurement))

# Expected Delivery Date
procurement["Expected_Delivery_Date"] = (
    procurement["Order_Date"] +
    pd.to_timedelta(lead_time, unit="D")
)

# Actual Delivery Date
procurement["Actual_Delivery_Date"] = pd.NaT

delivered_mask = procurement["Order_Status"] == "Delivered"

procurement.loc[
    delivered_mask,
    "Actual_Delivery_Date"
] = procurement.loc[
    delivered_mask,
    "Expected_Delivery_Date"
]

# Save back to database
save_to_db(
    procurement,
    "procurement_orders",
    "procurements_data"
)

print("Expected and Actual Delivery Dates populated successfully.")

Expected and Actual Delivery Dates populated successfully.


In [19]:
inventory_df = load_df.load("inventory")
forecast_df = load_df.load("forecast")

In [20]:
import pandas as pd
import numpy as np

def label_data() -> pd.DataFrame:
        """
        Compare current inventory against next-period forecasted demand and
        classify every product into a stockout risk bucket.

        FIX: original merge used `on=forecast['unique_id']` which passes a
        Series instead of a column name -- pandas silently mishandles this and
        drops/duplicates rows. Using the column name directly instead.
        """

        stockout = inventory_df.merge(
            right= forecast_df,
            on= forecast_df['unique_id'],
            how="left",
        )[["unique_id", "Current_Stock", "Reorder_Level", "Safety_Stock", "CrostonClassic"]]

        stockout["stockout"] = (stockout["Current_Stock"] - stockout["CrostonClassic"]).round()

        conditions = [
            # Already at/below safety stock
            stockout["Current_Stock"] <= stockout["Safety_Stock"],

            # Below reorder level but above safety stock
            (
                (stockout["Current_Stock"] > stockout["Safety_Stock"])
                & (stockout["Current_Stock"] <= stockout["Reorder_Level"])
            ),

            # Above reorder level now, but forecasted demand will push it
            # into safety stock before the next cycle
            (
                (stockout["Current_Stock"] > stockout["Reorder_Level"])
                & (
                    stockout["Current_Stock"] - stockout["CrostonClassic"]
                    <= stockout["Safety_Stock"]
                )
            ),

            # Healthy: stays above safety stock even after forecasted demand
            (
                (stockout["Current_Stock"] > stockout["Reorder_Level"])
                & (
                    stockout["Current_Stock"] - stockout["CrostonClassic"]
                    > stockout["Safety_Stock"]
                )
            ),
        ]

        choices = ["CRITICAL", "REORDER_NOW", "AT_RISK", "SUFFICIENT"]

        # Anything that doesn't match a rule (e.g. NaNs from an unmatched
        # product after the merge) is surfaced as NEEDS_REVIEW rather than
        # silently defaulting to a "safe" label.
        stockout["stockout_label"] = np.select(conditions, choices, default="NEEDS_REVIEW")

        save_to_db(stockout, "labeled_inventory_data", "master_data")

        return stockout

label_data()

,unique_id,Current_Stock,Reorder_Level,Safety_Stock,CrostonClassic,stockout,stockout_label
0,PROD00001,7,6,3,3.465759,4.0,SUFFICIENT
1,PROD00002,12,8,6,2.982145,9.0,SUFFICIENT
2,PROD00003,1,6,3,3.607468,-3.0,CRITICAL
3,PROD00004,10,9,2,3.676594,6.0,SUFFICIENT
4,PROD00005,6,6,2,3.287539,3.0,REORDER_NOW
...,...,...,...,...,...,...,...
195,PROD00196,6,7,3,3.826658,2.0,REORDER_NOW
196,PROD00197,6,6,5,2.901883,3.0,REORDER_NOW
197,PROD00198,13,6,5,2.629620,10.0,SUFFICIENT
198,PROD00199,18,7,2,3.078078,15.0,SUFFICIENT


In [21]:
labeled_inventory['Current_Stock'] = inventory_df['Current_Stock']

In [22]:
labeled_inventory = load_df.load("labeled_inventory")

In [23]:
labeled_inventory

,unique_id,Current_Stock,Reorder_Level,Safety_Stock,CrostonClassic,stockout,stockout_label
0,PROD00001,7,6,3,3.465759,4.0,SUFFICIENT
1,PROD00002,12,8,6,2.982145,9.0,SUFFICIENT
2,PROD00003,1,6,3,3.607468,-3.0,CRITICAL
3,PROD00004,10,9,2,3.676594,6.0,SUFFICIENT
4,PROD00005,6,6,2,3.287539,3.0,REORDER_NOW
...,...,...,...,...,...,...,...
195,PROD00196,6,7,3,3.826658,2.0,REORDER_NOW
196,PROD00197,6,6,5,2.901883,3.0,REORDER_NOW
197,PROD00198,13,6,5,2.629620,10.0,SUFFICIENT
198,PROD00199,18,7,2,3.078078,15.0,SUFFICIENT


In [24]:
from utils.db_crud import save_to_db

In [25]:
save_to_db(labeled_inventory, "labeled_inventory_data", "master_data")

In [ ]:

        if simulation:
            sim_eval = SimulationEvaluation()

            metrics = sim_eval.run_evaluation(self.run_id)

            logging.info("Simulaiton training metrics\n",metrics)
            
            return metrics

In [28]:
forecast_df = load_df.load("forecast")
forecast_df

,unique_id,ds,CrostonClassic
0,PROD00001,2024-09-29,3.465759
1,PROD00002,2024-09-29,2.982145
2,PROD00003,2024-09-29,3.607468
3,PROD00004,2024-09-29,3.676594
4,PROD00005,2024-09-29,3.287539
...,...,...,...
195,PROD00196,2024-09-29,3.826658
196,PROD00197,2024-09-29,2.901883
197,PROD00198,2024-09-29,2.629620
198,PROD00199,2024-09-29,3.078078


In [6]:
from backend.services.db_service import Load_Data
from utils.db_crud import save_to_db
load_df = Load_Data()

def update_zero_demand_ratio():
            # Reload latest simulation history
        load_df = Load_Data()
        sales = load_df.load("processed_sales")

        zero_ratio = (
            sales.groupby("unique_id")["y"]
            .apply(lambda x: (x == 0).mean())
            .reset_index(name="zero_demand_ratio")
        )

        sales = (
            sales.drop(columns=["zero_demand_ratio"], errors="ignore")
            .merge(
                zero_ratio,
                on="unique_id",
                how="left"
            )
        )

        save_to_db(
            sales,
            "processed_sales_orders",
            "processed_data"
        )
update_zero_demand_ratio()

In [27]:
sim = load_df.load("simulated_daily_sales")

In [29]:
sim.columns

Index(['Sales_Date', 'Product_ID', 'Opening', 'Forecast', 'Demand',
       'Sales_qty', 'Lost_sales', 'Closing'],
      dtype='object')

In [30]:
sup = load_df.load("suppliers")

In [31]:
sup.columns

Index(['Supplier_ID', 'Supplier_Name', 'Country', 'Region',
       'On_Time_Delivery_Rate', 'Certification_Level',
       'Is_Preferred_Supplier'],
      dtype='object')